# PPKS - Laboratorijska vježba 4: Digitalna modulacija

**Programska potpora komunikacijskim sustavima** | 2025./2026.

---

## Struktura vježbe

| **Dio** | **Sadržaj** | **Način rada** |
|---|---|---|
| 1 | Nosivi signal | simulacija |
| 2 | BPSK | simulacija |
| 3 | QPSK | simulacija |
| 4 | 16-QAM | simulacija |
| 5 | PlutoSDR | hardver |

**Dijelovi 1-4** pokreću se bez ikakve opreme, odnosno sve se simulira u Pythonu.
**Dio 5** zahtijeva spojeni PlutoSDR uređaj.

---

## Upute

Pokrenite ćelije redom od vrha prema dnu. Za svaki dio pogledajte output i razmislite o dobivenim rezultatima. Preporučujemo da mijenjate parametre (razina šuma, broj bitova, fazni pomak) i pratite kako se output mijenja.

---

## Potrebne instalacije

**Dijelovi 1–4** (simulacija) zahtijevaju standardne Python pakete:
```
pip install numpy matplotlib
```
**Dio 5** (PlutoSDR) zahtijeva nekoliko koraka:

**Korak 1 - USB driveri** - omogućuju OS-u prepoznavanje uređaja  
Preuzeti i pokrenuti installer **prije spajanja uređaja**:
- **Windows:** https://wiki.analog.com/university/tools/pluto/drivers/windows
- **macOS:** https://wiki.analog.com/university/tools/pluto/drivers/osx
- **Linux:** https://wiki.analog.com/university/tools/pluto/drivers/linux

**Korak 2 - libiio** - biblioteka za komunikaciju s PlutoSDR hardverom  
- **Windows:** preuzeti i pokrenuti [`libiio-0.26.ga0eca0d2-setup.exe`](https://github.com/analogdevicesinc/libiio/releases/download/v0.26/libiio-0.26.ga0eca0d2-setup.exe)

**Korak 3 - pyadi-iio** - Python sučelje koje koristimo u kodu; oslanja se na `libiio`
```
pip install pyadi-iio
```

## Provjera 

Spojiti PlutoSDR USB kabelom i u naredbenom retku pokrenuti:
```
ping 192.168.2.1
```
Ako stižu odgovori, uređaj je ispravno prepoznat i spreman za korištenje.


## Postavljanje parametara

Sljedeća ćelija definira zajedničke parametre koji se koriste kroz cijelu vježbu.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import adi
import time

# ── Parametri ─────────────────────────────────────────────────────
FS  = 1_000_000       # brzina uzorkovanja: 1 000 000 uzoraka/s
FC  = 2_400_000_000   # centralna frekvencija: 2.4 GHz 
SPS = 8               # uzoraka po simbolu
N   = 4096            # bitova po burstu
SCALE = 2**14         # koristimo za pretvorbu Python float raspona [-1, 1] u int16 raspon koji PlutoSDR ocekuje

np.random.seed(42)   

# Izvedeni parametri
SR = FS / SPS
print(f"Symbol rate : {SR/1e3:.0f} kBd  (koliko simbola šaljemo u sekundi)")
print(f"Bit rate    : {SR/1e3:.0f} kbps  (BPSK, 1 bit/simbol)")
print(f"Bit rate    : {2*SR/1e3:.0f} kbps  (QPSK, 2 bita/simbol)")
print(f"Bit rate    : {4*SR/1e3:.0f} kbps  (16-QAM, 4 bita/simbol)")
print(f"Bandwidth   : ~{SR/1e3:.0f} kHz  (null-to-null, isti za sve tri sheme)")


# -- Detekcija PlutoSDR-a ----------------------------------------------
try:
    sdr = adi.Pluto("ip:192.168.2.1")  # Windows i Linux
    #sdr = adi.Pluto("usb:0.1.5")      # macOS (ako IP adresa ne radi, provjerite USB adresu u terminalu)
    sdr.sample_rate           = int(FS)
    sdr.tx_lo                 = int(FC)
    sdr.rx_lo                 = int(FC)
    sdr.rx_buffer_size = 4 * N * SPS   # broj uzoraka koji se primaju u jednom pozivu sdr.rx()
    HARDWARE = True
    print("\nPlutoSDR pronaden - hardverski nacin rada.")
except Exception:
    sdr      = None
    HARDWARE = False
    print("\nPlutoSDR nije pronaden - simulacijski nacin rada.")


## Pomoćne funkcije

Ovdje su definirane sve funkcije koje se koriste kroz cijelu vježbu.

In [ ]:
def transceive(tx_signal):
    """
    Predaje i prima signal - identican poziv u simulaciji i na PlutoSDR hardveru.

    U simulaciji dodaje fazni pomak i AWGN sum kako bi signal izgledao realisticnije.
    """

    n_korisni = len(tx_signal)

    if HARDWARE:
        # 1. Unistimo eventualni stari TX buffer iz prethodnog poziva
        try:
            sdr.tx_destroy_buffer()
            time.sleep(0.3)
        except Exception:
            pass

        # 2. Praznimo RX FIFO - sve sto je hardver primao tijekom emitiranja
        #    starog signala jos uvijek je u bufferu i mora se ispustiti
        try:
            for _ in range(3):
                _ = sdr.rx()
        except Exception:
            pass

        # 3. Postavljamo novi TX signal
        sdr.tx_cyclic_buffer = True
        sdr.tx((tx_signal * SCALE).astype(np.complex64))
        time.sleep(0.8)

        # 4. Odbacujemo prvi RX buffer (mozda jos sadrzi ostatke prijelaza)
        try:
            _ = sdr.rx()
        except Exception:
            pass

        # 5. Primamo svjezi buffer s novim TX signalom
        rx_raw = sdr.rx().astype(np.complex64)
        sdr.tx_destroy_buffer()
        time.sleep(0.2)

        # Normalizacija amplitude na raspon [-1, 1]
        amp = np.max(np.abs(rx_raw))
        if amp > 0:
            rx_raw = rx_raw / amp

        return rx_raw.astype(np.complex64)
    else:
        fazni_pomak = np.exp(1j * 0.25)
        sum_sig = 0.1 * (np.random.randn(n_korisni) + 1j * np.random.randn(n_korisni))
        return tx_signal * fazni_pomak + sum_sig



def plot_spectrum(signal, fs, title="", ax=None):
    """Crta spektar snage signala u dBFS (decibeli u odnosu na puni opseg)."""
    n   = len(signal)
    f   = np.fft.fftfreq(n, d=1/fs)
    psd = 20 * np.log10(np.abs(np.fft.fft(signal)) / n + 1e-12)
    if ax is None:
        fig, ax = plt.subplots()
    ax.plot(np.fft.fftshift(f) / 1e3, np.fft.fftshift(psd))
    ax.set_xlabel("Frekvencijski pomak (kHz)")
    ax.set_ylabel("Snaga (dBFS)")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)



def bpsk_mod(bits, sps):
    """
    BPSK modulacija: bit 0 → -1, bit 1 → +1, svaki simbol traje sps uzoraka.

    Koristimo NRZ (Non-Return-to-Zero) kodiranje: 2*b - 1 preslikava {0,1} → {-1,+1}.

    Vidi: Mathuranathan, Digital Modulations using Python, pogl. 2.2.1
    """
    simboli = (2 * bits - 1).astype(float) 
    return np.repeat(simboli, sps).astype(np.complex64) # np.repeat() realizira pravokutne impulse duljine sps


def qpsk_mod(bits, sps):
    """
    QPSK modulacija: parovi bitova mapiraju se na 4 kuta IQ ravnine.

    Mapiranje (Gray kod): I = 1-2*b₀,  Q = 1-2*b₁,  |simbol| = 1.

    Vidi: Mathuranathan, Digital Modulations using Python, pogl. 2.5.1
    """
    bits_par = bits[: len(bits) - len(bits) % 2]
    b        = bits_par.reshape(-1, 2)
    simboli  = ((1 - 2 * b[:, 0]) + 1j * (1 - 2 * b[:, 1])) / np.sqrt(2)
    return np.repeat(simboli, sps).astype(np.complex64)


# Gray-kodirani amplitudni nivoi za 16-QAM (jedna os):
# 00→-3, 01→-1, 11→+1, 10→+3  (svake dvije susjedne točke dijele 1 bit razlike)
GRAY_MAP_1D = {(0, 0): -3, (0, 1): -1, (1, 1): 1, (1, 0): 3}


def qam16_mod(bits, sps):
    """
    16-QAM modulacija s Gray kodiranjem.

    Bitovi se grupiraju u nibble (4 bita): prva 2 → I os, druga 2 → Q os.
    Gray kodiranje: pogrešno prepoznata susjedna točka uzrokuje grešku u samo jednom bitu.

    Vidi: Mathuranathan, Digital Modulations using Python, pogl. 3.4.3
    """
    bits_p = bits[: len(bits) - len(bits) % 4]
    nibble = bits_p.reshape(-1, 4)
    simboli = np.array([
        GRAY_MAP_1D[tuple(n[:2])] + 1j * GRAY_MAP_1D[tuple(n[2:])]
        for n in nibble
    ])
    simboli /= (3 * np.sqrt(2))    # Normalizacija: najveća točka (3+3j) ima amplitudu 3√2 → dijeljenjem s 3√2 svodimo na 1.
    return np.repeat(simboli, sps).astype(np.complex64)


def sample_symbols(signal, sps):
    """Uzorkuje signal jednom po simbolu u trenutku optimalnog SNR-a (sredina simbola)."""
    return signal[sps // 2 :: sps]


def add_awgn(signal, snr_db):
    """
    Dodaje AWGN (aditivni bijeli Gaussov šum) pri zadanom SNR-u izraženom u dB.

    AWGN je standardni model koji simulira termički šum prijemnika.
    Snaga šuma izračunava se iz snage signala i zadanog SNR-a:
        P_noise = P_signal / SNR_linear

    Vidi: Mathuranathan, Digital Modulations using Python, pogl. 4.1.2
    """
    snr_lin  = 10 ** (snr_db / 10)
    p_signal = np.mean(np.abs(signal) ** 2)
    p_noise  = p_signal / snr_lin
    šum      = np.sqrt(p_noise / 2) * (
        np.random.randn(*signal.shape) + 1j * np.random.randn(*signal.shape)
    )
    return signal + šum


print("Sve funkcije su učitane.")

# Generiramo zajednički niz bitova - koristit će se kroz cijelu vježbu
bits_tx = np.random.randint(0, 2, N)
print(f"Generirano {N} slučajnih bitova za slanje.")


---
## Dio 1 - Nosivi signal (*carrier*)

Nosivi signal je nemodulirani sinusni signal određene frekvencije koji sam po sebi ne nosi informacije. Modulacijom se bitovi kodiraju u promjene njegove amplitude, faze ili frekvencije.

Ovaj korak prikazuje kako nemodulirani nosivi signal izgleda u vremenskoj i frekvencijskoj domeni. 

In [ ]:
t       = np.arange(2048) / FS                                      # vremenski vektor, 2048 uzoraka
carrier = np.exp(1j * 2 * np.pi * 100e3 * t).astype(np.complex64)   # nosivi signal na +100 kHz 

# Dodajemo Gaussov šum - imitira termički šum prijemnika
šum     = 0.02 * (np.random.randn(2048) + 1j * np.random.randn(2048))
rx      = carrier + šum

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Lijevi graf: I i Q u vremenskoj domeni (prvih 60 uzoraka ≈ 6 perioda nosivog signala)
ax1.plot(t[:60] * 1e6, rx[:60].real, label="I (realna komponenta)",  lw=1.8)
ax1.plot(t[:60] * 1e6, rx[:60].imag, label="Q (imaginarna komponenta)", alpha=0.75, lw=1.8)
ax1.set_xlabel("Vrijeme (μs)")
ax1.set_ylabel("Amplituda")
ax1.set_title("Nosivi signal - vremenska domena")
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Desni graf: spektar 
plot_spectrum(rx, FS, title="Nosivi signal - spektar", ax=ax2)
ax2.set_ylim(-80, 5)

plt.tight_layout()
plt.show()

### Što vidimo

**Lijevi graf - vremenska domena**
- I i Q su sinusoide iste frekvencije i amplitude, pomaknute za 90°
- I i Q zajedno opisuju amplitudu i fazu: A = √(I²+Q²), φ = arctan(Q/I)

**Desni graf - spektar**
- šiljak na +100 kHz ≈ 0 dBFS (nosivi signal je dominantan signal u spektru)
- šum ≈ -60 dBFS po cijelom spektru

**dBFS** = decibeli relativno na *Full Scale* - 0 dBFS je maksimum koji sustav može reprezentirati, sve niže je negativno.

---
## Dio 2 - BPSK modulacija

**Binary Phase Shift Keying (BPSK)** je najjednostavnija digitalna modulacija: faza nosivog signala skače za 180° ovisno o bitu.

| **Bit** | **Simbol** | **Faza** |
|---|---|---|
| 0 | −1 | 180° |
| 1 | +1 | 0° |

Jedan simbol nosi jedan bit → bit rate = symbol rate.

**Gdje se BPSK koristi?**
- **GPS i satelitska navigacija** (Galileo, GLONASS) - signal putuje tisuće kilometara te je robustnost prioritet nad brzinom
- **Deep space komunikacija** (NASA, ESA) - npr. signali s Voyager 1 i 2 putuju milijardama kilometara → pri tako niskoj razini signala složenija modulacija ne bi radila pouzdano
- **WiFi (802.11a/g)** - pri slabom signalu, na rubu pokrivenosti, sustav automatski prelazi na BPSK
- **RFID (beskontaktne kartice i putovnice)** - pasivni odašiljač nema puno snage → BPSK je dovoljno robustan

BPSK se koristi kada je SNR nizak i kada je pouzdanost prijenosa važnija od brzine.

In [ ]:
bpsk_sig = bpsk_mod(bits_tx, SPS)

# Simuliramo kanal: fazni pomak 0.25 rad (≈14°) + AWGN 
šum_bpsk = 0.1 * (np.random.randn(len(bpsk_sig)) + 1j * np.random.randn(len(bpsk_sig)))
rx_bpsk  = bpsk_sig * np.exp(1j * 0.25) + šum_bpsk

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- 1. Vremenska domena: prvih 20 simbola ---
t_us = np.arange(SPS * 20) / FS * 1e6
axes[0].plot(t_us, bpsk_sig[:SPS * 20].real, label="Tx (idealni)",  lw=2, color="steelblue")
axes[0].plot(t_us, rx_bpsk[:SPS * 20].real,  label="Rx (sa šumom)", alpha=0.7, color="tomato")
axes[0].axhline(0, color="k", lw=0.5)
# Označavamo granice simbola radi preglednosti
for k in range(20):
    axes[0].axvline(k * SPS / FS * 1e6, color="gray", lw=0.4, ls="--")
axes[0].set(title="Vremenska domena (20 simbola)", xlabel="Vrijeme (μs)", ylabel="Amplituda")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# --- 2. Spektar ---
plot_spectrum(rx_bpsk[:2048], FS, title="Spektar", ax=axes[1])
axes[1].set_ylim(-80, 5)

# --- 3. Konstelacijski dijagram: uzorkujemo jednom po simbolu ---
# Uzorkujemo u sredini simbola (sps//2). Tamo je signal najstabilniji (daleko od rubova simbola).
syms_bpsk = sample_symbols(rx_bpsk, SPS)
axes[2].scatter(syms_bpsk.real, syms_bpsk.imag, alpha=0.35, s=12, color="steelblue")
axes[2].axhline(0, color="k", lw=0.5)
axes[2].axvline(0, color="k", lw=0.5)
# Crtamo idealne konstelacijske točke za BPSK (+1 i -1 na I osi)
axes[2].scatter([-1, 1], [0, 0], color="red", s=80, zorder=5, marker="x", lw=2,
                label="Idealne točke")
axes[2].set(title="Konstelacijski dijagram", xlabel="I", ylabel="Q", xlim=(-2, 2), ylim=(-2, 2))
axes[2].set_aspect("equal")
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.suptitle("BPSK modulacija", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Generirano {N} BPSK simbola | Symbol rate: {FS/SPS/1e3:.0f} kBd | Bit rate: {FS/SPS/1e3:.0f} kbps")

### Što vidimo

**Lijevi graf - vremenska domena**
- Plavi signal (Tx) skače između +1 i -1. Svaki simbol odgovara jednom bitu
- Narančasti signal (Rx) je isti signal s dodanim šumom i faznim pomakom

**Srednji graf - spektar**
- Nakon modulacije signal se raspoređuje po frekvencijama i umjesto jednog uskog šiljka zauzima široki pojas
- Oblik spektra je sinc funkcija (posljedica pravokutnih impulsa u vremenskoj domeni)
- Širina spektra ovisi isključivo o symbol rate-u, ne o sadržaju poruke (bandwidth je isti bez obzira šaljemo li same nule, same jedinice ili mješavinu)

**Desni graf - konstelacijski dijagram**
- Dva oblaka odgovaraju bitu 0 (lijevo) i bitu 1 (desno)
- Oblaci su zarotirani zbog faznog pomaka dodanog u simulaciji, a raspršenost oblaka odražava razinu šuma u kanalu

---
## Dio 3 - Usporedba BPSK i QPSK

**Quadrature Phase Shift Keying (QPSK)** kodira 2 bita po simbolu koristeći četiri faze umjesto dvije. Pri istom symbol rateu (125 kBd), QPSK postiže dvostruku brzinu prijenosa bita  (250 kbps) **u istom bandwidthu**.

| **Modulacija** | **Bitova/simbol** | **Bit rate** | **Faze** |
|---|---|---|---|
| BPSK | 1 | 125 kbps | 0°, 180° |
| QPSK | 2 | 250 kbps | 45°, 135°, 225°, 315° |

**Gray kodiranje:** parovi bitova (`00`, `01`, `10`, `11`) raspoređeni su po 
kutovima IQ ravnine tako da se svake dvije susjedne točke razlikuju u samo 1 bitu. 
Kad šum pogura simbol na pogrešnu susjednu točku, griješimo u samo jednom bitu umjesto u dva.

**Gdje se QPSK koristi?**

- **Satelitska televizija (DVB-S2)** - signal putuje 36 000 km do Zemlje te je SNR nizak; QPSK je otporniji na šum od viših modulacija.
- **GNSS (GPS, Galileo...)** - temeljna modulacijska tehnika u navigacijskim sustavima, gdje pouzdanost prijema ima prioritet.
- **Mobilne mreže i Wi-Fi** - koristi se u 4G LTE, 5G kontrolnim kanalima i IEEE 802.11 standardima; na rubovima pokrivenosti gdje složenije modulacije nisu pouzdane.

In [ ]:
# Koristimo bpsk_sig iz prethodne ćelije te generiramo QPSK signal.
# Pritom, fazni pomak i razina šuma su isti za oba signala (kako bismo ih mogli usporediti).

qpsk_sig = qpsk_mod(bits_tx, SPS)

fazni_pomak = np.exp(1j * 0.25)   # ~14° - imitira CFO prijemnika
šum_komp    = 0.10   # isti za BPSK i QPSK radi usporedbe

šum_b = šum_komp * (np.random.randn(len(bpsk_sig)) + 1j * np.random.randn(len(bpsk_sig)))
šum_q = šum_komp * (np.random.randn(len(qpsk_sig)) + 1j * np.random.randn(len(qpsk_sig)))

rx_bpsk2 = bpsk_sig * fazni_pomak + šum_b
rx_qpsk  = qpsk_sig * fazni_pomak + šum_q

syms_bpsk2 = sample_symbols(rx_bpsk2, SPS)
syms_qpsk  = sample_symbols(rx_qpsk,  SPS)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# --- 1. BPSK konstelacija ---
axes[0].scatter(syms_bpsk2.real, syms_bpsk2.imag, alpha=0.35, s=12, color="steelblue")
# Idealne točke (bez faznog pomaka)
axes[0].scatter([-1, 1], [0, 0], color="red", s=80, zorder=5, marker="x", lw=2)
axes[0].axhline(0, color="k", lw=0.5); axes[0].axvline(0, color="k", lw=0.5)
axes[0].set_title("BPSK konstelacijski dijagram")
axes[0].set_xlabel("I"); axes[0].set_ylabel("Q")
axes[0].set_aspect("equal"); axes[0].set_xlim(-2, 2); axes[0].set_ylim(-2, 2)
axes[0].grid(True, alpha=0.3)
# Anotacija (koji oblak odgovara kojem bitu)
axes[0].text(-1.0, -1.6, "bit 0", ha="center", fontsize=9, color="steelblue")
axes[0].text( 1.0, -1.6, "bit 1", ha="center", fontsize=9, color="steelblue")

# --- 2. QPSK konstelacija ---
axes[1].scatter(syms_qpsk.real, syms_qpsk.imag, alpha=0.35, s=12, color="tomato")
# Idealne QPSK točke (bez faznog pomaka)
ideal_qpsk = np.array([1+1j, -1+1j, -1-1j, 1-1j]) / np.sqrt(2) 
axes[1].scatter(ideal_qpsk.real, ideal_qpsk.imag, color="darkred", s=80,
                zorder=5, marker="x", lw=2)
axes[1].axhline(0, color="k", lw=0.5); axes[1].axvline(0, color="k", lw=0.5)
axes[1].set_title("QPSK konstelacijski dijagram")
axes[1].set_xlabel("I"); axes[1].set_ylabel("Q")
axes[1].set_aspect("equal"); axes[1].set_xlim(-2, 2); axes[1].set_ylim(-2, 2)
axes[1].grid(True, alpha=0.3)
# Označavanje kojem kutu odgovaraju koji bitovi (prije faznog pomaka)
for txt, pos in [("00", (-1.55, 1.35)), ("10", (1.25, 1.35)),
                 ("01", (-1.55,-1.55)), ("11", (1.25,-1.55))]:
    axes[1].text(pos[0], pos[1], txt, fontsize=9, color="darkred", fontweight="bold")

# --- 3. Spektar: BPSK vs QPSK  ---
freqs   = np.fft.fftfreq(2048, d=1/FS)
psd_b   = 20 * np.log10(np.abs(np.fft.fft(rx_bpsk2[:2048])) / 2048 + 1e-12)
psd_q   = 20 * np.log10(np.abs(np.fft.fft(rx_qpsk[:2048]))  / 2048 + 1e-12)

axes[2].plot(np.fft.fftshift(freqs)/1e3, np.fft.fftshift(psd_b),
             label="BPSK", alpha=0.85, lw=1.5, color="steelblue")
axes[2].plot(np.fft.fftshift(freqs)/1e3, np.fft.fftshift(psd_q),
             label="QPSK", alpha=0.85, lw=1.5, color="tomato", ls="--")
axes[2].set(title="Spektar",
            xlabel="Frekvencijski pomak (kHz)", ylabel="Snaga (dBFS)")
axes[2].set_ylim(-80, 5)
axes[2].legend()
axes[2].grid(True, alpha=0.3)


plt.suptitle("BPSK vs QPSK", fontsize=13)
plt.tight_layout()
plt.show()

### Što vidimo?

**BPSK konstelacijski dijagram** - Dva oblaka su zarotirana zbog simuliranog faznog pomaka. Maksimalan razmak između dvije točke znači najveću otpornost na šum.

**QPSK konstelacijski dijagram** - Četiri oblaka raspoređena su u kutovima IQ ravnine, zarotirana od idealnih pozicija (±45°, ±135°) zbog istog faznog pomaka. Manji razmak između susjednih simbola znači veću osjetljivost na šum u odnosu na BPSK.

**Spektar** - Krivulje se preklapaju jer bandwidth ovisi isključivo o symbol rateu, ne o broju bitova po simbolu. Isti symbol rate znači isti bandwidth, no QPSK prenosi dvostruko više podataka u istom frekvencijskom prostoru.

---
## Dio 4 - Viši redovi modulacije: 16-QAM

Do sada smo mijenjali samo fazu nosivog signala. **Quadrature Amplitude Modulation (QAM)** neovisno mijenja amplitudu I i Q komponente, što omogućuje više konstelacijskih točaka i veći bit rate (brzina prijenosa bita) pri istom symbol rateu i bandwidthu.

| **Modulacija** | **Stanja** | **Bitova/simbol** | **Bit rate** |
|---|---|---|---|
| BPSK | 2 | 1 | 125 kbps |
| QPSK | 4 | 2 | 250 kbps |
| 16-QAM | 16 | 4 | 500 kbps |
| 64-QAM | 64 | 6 | 750 kbps |
| 256-QAM | 256 | 8 | 1000 kbps |

**Gdje se QAM koristi?**  
- **WiFi 6/6E (802.11ax)** - do 1024-QAM  
- **4G LTE / 5G** - do 256-QAM na downlinku  
- **Kabelska televizija (DVB-C)** - 256-QAM

In [ ]:
# Prikazujemo sve tri modulacije pri istom SNR-u radi usporedbe,
# i 16-QAM pri tri SNR-a da vidimo kako šum utječe na čitljivost.

SNR_USPOREDBA = 18   # dB 

# Moduliramo s istim bits_tx
rx_bpsk_q = add_awgn(bpsk_mod(bits_tx, SPS),  SNR_USPOREDBA)
rx_qpsk_q = add_awgn(qpsk_mod(bits_tx, SPS),  SNR_USPOREDBA)
rx_qam_q  = add_awgn(qam16_mod(bits_tx, SPS), SNR_USPOREDBA)

syms_b = sample_symbols(rx_bpsk_q, SPS)
syms_q = sample_symbols(rx_qpsk_q, SPS)
syms_m = sample_symbols(rx_qam_q,  SPS)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# ── Red 1: usporedba modulacija pri istom SNR-u ───────────────────────────────
fig.text(0.5, 0.92, f"Usporedba modulacija pri SNR = {SNR_USPOREDBA} dB",
         ha="center", fontsize=12, fontweight="bold")

for ax, syms, naziv, boja in zip(
        axes[0],
        [syms_b, syms_q, syms_m],
        ["BPSK - 1 bit/sim", "QPSK - 2 bita/sim", "16-QAM - 4 bita/sim"],
        ["steelblue", "tomato", "seagreen"]):
    ax.scatter(syms.real, syms.imag, alpha=0.3, s=12, color=boja)
    ax.axhline(0, color="k", lw=0.5); ax.axvline(0, color="k", lw=0.5)
    ax.set(title=naziv, xlabel="I", ylabel="Q", xlim=(-2, 2), ylim=(-2, 2))
    ax.set_aspect("equal"); ax.grid(True, alpha=0.3)
    if "QAM" in naziv:
        for _, ai in GRAY_MAP_1D.items():
            for _, aq in GRAY_MAP_1D.items():
                ax.scatter(ai / (3*np.sqrt(2)), aq / (3*np.sqrt(2)),
                           color="darkgreen", s=30, zorder=5, marker="x", lw=1)
    elif "QPSK" in naziv:
        ideal_qpsk = np.array([1+1j, -1+1j, -1-1j, 1-1j]) / np.sqrt(2)
        ax.scatter(ideal_qpsk.real, ideal_qpsk.imag,
                   color="darkred", s=80, zorder=5, marker="x", lw=2)
    elif "BPSK" in naziv:
        ax.scatter([-1, 1], [0, 0],
                   color="red", s=80, zorder=5, marker="x", lw=2)

# ── Red 2: 16-QAM pri trima SNR razinama ─────────────────────────────────────
fig.text(0.5, 0.44, "16-QAM pri različitim razinama SNR-a",
         ha="center", fontsize=12, fontweight="bold")

for ax, snr_prik, boja in zip(axes[1], [5, 15, 25], ["coral", "goldenrod", "seagreen"]):
    rx_prik   = add_awgn(qam16_mod(bits_tx, SPS), snr_prik)
    syms_prik = sample_symbols(rx_prik, SPS)
    ax.scatter(syms_prik.real, syms_prik.imag, alpha=0.35, s=12, color=boja)
    ax.axhline(0, color="k", lw=0.5); ax.axvline(0, color="k", lw=0.5)
    for _, ai in GRAY_MAP_1D.items():
        for _, aq in GRAY_MAP_1D.items():
            ax.scatter(ai / (3*np.sqrt(2)), aq / (3*np.sqrt(2)),
                       color="k", s=25, zorder=5, marker="+", lw=1.2)
    ax.set(title=f"SNR = {snr_prik} dB",
           xlabel="I", ylabel="Q", xlim=(-1.5, 1.5), ylim=(-1.5, 1.5))
    ax.set_aspect("equal"); ax.grid(True, alpha=0.3)
    # Granice odluke - dijele IQ ravninu na 16 zona, po jedna za svaku konstelacijsku točku
    for v in [-2/(3*np.sqrt(2)), 0, 2/(3*np.sqrt(2))]:
        ax.axhline(v, color="gray", lw=0.5, ls=":")
        ax.axvline(v, color="gray", lw=0.5, ls=":")

plt.tight_layout(rect=[0, 0, 1, 0.90], h_pad=4.0)
plt.show()

### Što vidimo?

- **Gornji red** - sve tri modulacije su prikazane pri SNR = 18 dB. BPSK ima dva široko razmaknuta oblaka, QPSK četiri, 16-QAM šesnaest zbijenih u isti prostor. Veći kapacitet znači manji razmak između susjednih simbola i veću osjetljivost na šum.

- **Donji red** - 16-QAM pri SNR = 5, 15 i 25 dB. Pri 5 dB oblaci se šire i preklapaju s granicama odluke (svijetlo sive linije). Pri 25 dB svih 16 oblaka jasno je odvojeno oko idealnih pozicija.

---
## Dio 5 - PlutoSDR

Za ovaj dio potrebno je spojiti PlutoSDR uređaj USB kabelom s računalom. Također, prije pokretanja ovog dijela potrebno je pokrenuti ćelije s parametrima i pomoćnim funkcijama (nalaze se na početku bilježnice).

### Odabir modulacije i načina rada

Uredite parametre definirane na vrhu sljedeće ćelije:

| **Parametar** | **Opcije** | **Opis** |
|---|---|---|
| `MODULACIJA` | `"BPSK"`, `"QPSK"`, `"QAM16"` | Vrsta digitalne modulacije |
| `ULOGA` | `"loopback"`, `"tx"`, `"rx"` | Način rada uređaja |
| `TX_GAIN` | cijeli broj od -90 do 0 | Pojačanje predajnika u dB |
| `RX_GAIN` | cijeli broj od 0 do 74.5 | Pojačanje prijemnika u dB |

**Načini rada:**
- `loopback` - jedan PlutoSDR, TX i RX spojeni koaksijalnim kabelom
- `tx` - jedan od dva PlutoSDR-a, samo emitira signal
- `rx` - drugi PlutoSDR, samo prima signal

Za bežični prijenos između dva uređaja (over-the-air, OTA) pokrenuti `tx` na jednom računalu i `rx` na drugom.

**Preporuka za pojačanja:**
- Loopback: `TX_GAIN = -40`, `RX_GAIN = 30`
- OTA: `TX_GAIN = -20`, `RX_GAIN = 40`

---

### Zadaci

**1. Loopback - utjecaj pojačanja**

Postavite `ULOGA = "loopback"` i isprobajte različite kombinacije `TX_GAIN` i `RX_GAIN`. Promatrajte kako se mijenjaju konstelacijski dijagram i spektar. 

**2. Loopback - usporedba modulacija**

Zadržite iste gain postavke i mijenjajte `MODULACIJA` između `"BPSK"`, `"QPSK"` i `"QAM16"`. Usporedite konzistentnost rezultata između modulacija i razmislite zašto su neki rezultati stabilniji od drugih.

**3. OTA - bežični prijenos**

Radite u suradnji s drugom grupom (potrebna su dva PlutoSDR-a): jedan PlutoSDR ima `ULOGA = "tx"`, drugi `ULOGA = "rx"`. Oboje moraju imati istu vrijednost `MODULACIJA`. TX pokreće ćeliju prvi i čeka, RX pokreće ćeliju i promatra rezultat.

- Kako izgleda konstelacija u usporedbi s loopbackom?
- Mijenja li se išta između uzastopnih pokretanja?
- Koji je mogući razlog razlike između loopback i OTA rezultata?
- Kako poboljšati rezultate OTA prijenosa? 

In [ ]:
# -- Dio 5: PlutoSDR  --------------------------------------------------
MODULACIJA = "BPSK"       # "BPSK" | "QPSK" | "QAM16"
ULOGA      = "loopback"   # "loopback" | "tx" | "rx"
TX_GAIN    = -40           # Preporuka za loopback: od -40 do -50. Preporuka za OTA: od -20 do -10
RX_GAIN    = 30           # Preporuka za loopback: od 20 do 30. Preporuka za OTA: od 30 do 50
# -----------------------------------------------------------------------------

if "HARDWARE" not in dir():
    raise RuntimeError("Varijabla HARDWARE nije definirana. Pokrenite celiju s parametrima i celiju s pomocnim funkcijama.")

if not HARDWARE:
    raise RuntimeError("PlutoSDR nije detektiran. Spojite uredaj i pokrenite celiju s parametrima.")

sdr.tx_hardwaregain_chan0 = TX_GAIN
sdr.rx_hardwaregain_chan0 = RX_GAIN

mod_fn = {"BPSK": bpsk_mod, "QPSK": qpsk_mod, "QAM16": qam16_mod}[MODULACIJA]

if ULOGA == "loopback":
    tx_sig = mod_fn(bits_tx, SPS)
    rx_sig = transceive(tx_sig)

elif ULOGA == "tx":
    # Emitiranje traje dok ga rucno ne zaustavimo: Kernel -> Interrupt
    tx_sig = mod_fn(bits_tx, SPS)
    sdr.tx_cyclic_buffer = True
    sdr.tx((tx_sig * SCALE).astype(np.complex64))
    print(f"Emitiranje: {MODULACIJA} na {FC/1e9:.1f} GHz. Za zaustavljanje: Kernel -> Interrupt.")
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        sdr.tx_destroy_buffer()
        print("Emitiranje zaustavljeno.")

elif ULOGA == "rx":
    rx_sig = sdr.rx().astype(np.complex64)
    amp = np.max(np.abs(rx_sig))
    if amp > 0:
        rx_sig = rx_sig / amp

# Vizualizacija (loopback i rx)
if ULOGA in ("loopback", "rx"):
    nacin = f"{ULOGA} - PlutoSDR"

    ideal_pts = {
        "BPSK":  np.array([-1+0j, 1+0j]),
        "QPSK":  np.array([1+1j, -1+1j, -1-1j, 1-1j]) / np.sqrt(2),
        "QAM16": np.array([
            a/(3*np.sqrt(2)) + 1j*b/(3*np.sqrt(2))
            for a in [-3,-1,1,3] for b in [-3,-1,1,3]
        ])
    }

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    plot_spectrum(rx_sig[:2048], FS, title=f"{MODULACIJA} - spektar ({nacin})", ax=axes[0])
    axes[0].set_ylim(-80, 5)

    syms_rx = sample_symbols(rx_sig, SPS)
    axes[1].scatter(syms_rx.real, syms_rx.imag, alpha=0.1, s=10)
    axes[1].scatter(ideal_pts[MODULACIJA].real, ideal_pts[MODULACIJA].imag,
                    color="red", s=80, zorder=5, marker="x", lw=2, label="Idealne tocke")
    axes[1].axhline(0, color="k", lw=0.5); axes[1].axvline(0, color="k", lw=0.5)
    axes[1].set(title=f"{MODULACIJA} - konstelacija ({nacin})",
                xlabel="I", ylabel="Q", xlim=(-1.5, 1.5), ylim=(-1.5, 1.5))
    axes[1].set_aspect("equal"); axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.1)

    plt.suptitle(f"Dio 5 - {MODULACIJA} ({nacin})", fontsize=13)
    plt.tight_layout()
    plt.show()

    print(f"Nacin rada : {nacin}")
    print(f"Modulacija : {MODULACIJA}")
